# StartRight: checking startup states in SKY130 bias circuits

**Author: Adnane Essoufi. Solo project.**

A bias circuit can appear to start with one simulator setting and fail with another. StartRight studies that problem in a SKY130 inverse-Widlar circuit. It connects a gate-cut return curve to operating points in the intact circuit, then checks a useful current output against a startup requirement.

The topology and conventional startup device are established work. The contribution is an open verification workflow with measured examples. The same verification code also checks a beta multiplier and a resistor-biased mirror. These additional examples test reuse and give the workflow healthy circuits to assess.


## Circuit and scope

The five-device core follows Fig. 1 of Wang et al. (2012). We cut only the gate connection of M1 for the DC return curve. M2 remains diode-connected. The selected BSIM4 models disable DC gate leakage, and we check that current through the cut-gate voltage source is zero. The startup NMOS follows the connection in Fig. 8 of the same paper.

M5 uses W/L=12/1 Âµm. M4 uses 0.42/20 Âµm. M1, M2 and M3 use 4/1 Âµm. The startup device uses 0.42/1 Âµm. Each internal node has a declared 100 fF load.

![Circuit connections](results/loaded/circuit_connections.png) Rectangular junction geometry uses 0.29 Âµm diffusion extensions. Source/drain square-resistance parameters remain at zero. These are schematic assumptions, not extracted geometry.

The main example uses TT at 27 Â°C and a 1.8 V rail. It consumes about 112 ÂµA in the upper operating state. It is a verification example, not an optimized batteryless reference.


## Setup

The fresh short path has passed on Windows with Python 3.12.8 and in an Alpine Linux VM with Python 3.12.15, on the same computer. Both use ngspice 47 and the same pinned SKY130 models. Windows uses the official portable binary. Linux builds its binary from the checksum-pinned source archive.

For Windows, create an isolated environment, install `requirements-notebook-lock-windows-py312.txt` and run `python bootstrap.py`. Linux requires Python 3.12 and a C/C++ build toolchain (`gcc`, `g++`, `make`). Create an isolated environment, install `requirements-notebook-lock-linux-py312.txt`, then run `python bootstrap_linux.py --jobs 2`. With an existing ngspice 47 installation, Linux can use `bootstrap.py` instead. README.md gives the commands.

The bootstrap checks 24 model/license files. The source builder installs ngspice inside the project cache. It does not install operating-system packages. Colab and macOS have not been tested.

The live cells audit both DC sweep directions, confirm candidate points in the intact circuit and run a loaded cold start. `status=completed` means the analysis completed. Read `startup_verdict` for readiness. Wider qualification studies below are saved data with regeneration scripts.


## Why the test works

For the gate-cut circuit, let F(v) be the returned voltage at b when the disconnected M1 gate is held at v. A crossing of F(v) and v is a candidate operating point. We preserve the diode connections and output loading, and audit the cut-gate current before using the curve. The local slope is a diagnostic, not a proof of stability for an arbitrary circuit. We therefore check the candidates in the intact circuit and use perturbation tests.

At the nominal unwanted low point, b is about 0.018 V and a is about 1.406 V. The circuit draws less than a picoamp in this model configuration. Numerical junction conductance can compete with that weak current and change the return curve. At the desired point, the circuit draws about 112 ÂµA, so the same numerical conductance has little effect on the intended bias value.

The startup NMOS connects a to b with its gate tied to a. It conducts while a is high and b is low, helping the circuit leave the unwanted region. At the desired point a falls below b, so this startup path no longer strongly drives the bias node. These connections are established circuit practice. Our experiment measures their behavior with the stated open models and numerical checks.


In [1]:
from pathlib import Path
import sys, json
import numpy as np
PROJECT = Path.cwd()
if not (PROJECT / 'startright.py').exists():
    PROJECT = PROJECT / 'ISSCC27' / 'submitted_notebooks' / 'StartRight'
assert (PROJECT / '.startright_tools.json').exists(), 'Run bootstrap.py first.'
sys.path.insert(0, str(PROJECT.resolve()))
from startright import verify_case, verify_circuit
from circuits import CIRCUITS
from widlar_pilot import STUDY, audit_return_case
print('Python:', sys.version.split()[0], '; NumPy:', np.__version__)
print('Isolated Python environment:', sys.prefix != sys.base_prefix)
print('Pinned SKY130 commit: f62031a1be9aefe902d6d54cddd6f59b57627436')


Python: 3.12.8 ; NumPy: 2.3.5
Isolated Python environment: True
Pinned SKY130 commit: f62031a1be9aefe902d6d54cddd6f59b57627436


## Live return-curve experiment

This cell freshly verifies the unassisted circuit at two GMIN values. Each verification runs both sweep directions, intact-circuit checks and a loaded cold start. The input current check and complete row count are required before interpreting the crossings. GMIN is a numerical minimum conductance. The ngspice manual lists 1 pS as its default. We hold the other numerical settings fixed in this comparison.

A crossing is only a candidate operating point. The saved study below separately checks the three nominal candidates in the intact circuit. A finite grid can miss close or tangent roots.


In [2]:
live_reports = []
for gmin in [1e-12, 1e-15]:
    result = verify_case(None, gmin=gmin)
    assert result['status'] == 'completed'
    live_reports.append(result)
    print('GMIN', gmin, 'S:', [round(r['intact_b_V'], 6) for r in result['candidate_operating_points']])
    print('Loaded cold start:', result['startup_verdict'])


GMIN 1e-12 S: [1.160091]
Loaded cold start: fails_declared_bench


GMIN 1e-15 S: [0.017701, 0.118202, 1.160092]
Loaded cold start: fails_declared_bench


## Live cold-start experiment

The source begins at zero. It stays there for 1 Âµs, then rises to 1.8 V over 1 Âµs. No internal initial conditions force the result. The live loaded run ends at about 20 ms. Earlier saved bias-only runs use a longer interval.

The live loaded test targets the intended current output within 5%, with VDD at least 1.7 V and output compliance at least 0.3 V. It must reach the band within 1 ms after the ramp finishes and remain there through the run for at least 2 ms. The earlier saved bias-only study uses a separate voltage-band check. Neither contract claims current or voltage accuracy over PVT.


In [3]:
assisted = verify_case(0.42)
assert assisted['status'] == 'completed'
assert assisted['loaded_test']['meets_contract']
print(assisted['startup_verdict'])
print(json.dumps(assisted['loaded_test'], indent=2))


passes_declared_bench
{
  "case": "verify_inverse_widlar_2e86aaaf_loaded_cold",
  "nominal_target_A": 7.7316061e-07,
  "final_output_A": 7.73160419e-07,
  "final_bias_V": 1.16009189,
  "method": "last entry into the valid band, linearly interpolated between samples; valid through the end",
  "target_output_A": 7.7316061e-07,
  "deadline_s": 0.001002,
  "hold_s": 0.002,
  "ready_time_s": 1.809137679946822e-05,
  "source_energy_to_ready_J": 1.1752679913184876e-10,
  "meets_contract": true,
  "first_valid_sample_s": 1.80919833e-05
}


## Saved validation study

This section reads saved results. It does not rerun all conditions. Regenerate them with the commands in README.md.

The two DC sweep directions agreed within 0.1 nV in the nominal lower-GMIN case. All three candidates converged to matching operating points in the intact circuit. The largest bias-voltage mismatch was about 0.32 ÂµV.

The six diagnostic perturbations support the expected local behavior. The low-state perturbations had only partly relaxed in the 20 ms window. They should not be described as fully settled trajectories.

Eight cold-start checks vary GMIN, integration method and maximum time step. The assisted circuit reaches the nominal band; the unassisted lower-GMIN circuit remains at low bias. A separate screen uses five process corners and three temperatures. It finds one candidate root with startup in each tested condition, but it is not an exhaustive proof.


In [4]:
validation = json.loads((STUDY / 'validation.json').read_text())
print(json.dumps(validation, indent=2))
ops = json.loads((STUDY / 'intact_operating_points.json').read_text())
print('Nominal intact bias points (V):', [r['intact_b_V'] for r in ops])


{
  "gmin_1e-12_forward_reverse_max_V": 0.0,
  "gmin_1e-15_forward_reverse_max_V": 1.000000013351432e-10,
  "max_intact_root_error_V": 3.192251083888875e-07,
  "cold_start_numerical_cases_passed": 8,
  "condition_scans_completed": 30,
  "unassisted_conditions_with_3_candidate_roots": 8,
  "startup_conditions_with_1_candidate_root": 15,
  "invalid_contact_case_correctly_rejected": {
    "accepted": false,
    "reasons": [
      "Nonzero cut-gate current violates this unloaded gate-cut bench",
      "Transient operating-point fallback was used during the DC run"
    ],
    "points": 3601,
    "max_abs_cut_gate_current_A": 8.35614365e-09,
    "dc_transient_fallback": true
  },
  "excluded_contact_trial": {
    "reason": "DC solver invoked transient operating-point fallback and cut-gate current was nonzero; not accepted as a valid unloaded DC return curve",
    "max_abs_cut_gate_current_A": 8.35614365e-09
  },
  "scope": "Finite-grid model evidence with explicit junction geometry and defau

![Startup evidence](results/widlar/startup_evidence.png)

For this declared circuit, GMIN changes the operating-point picture and startup behavior. Successful startup at the default setting alone does not show that the unwanted state has been removed. The conventional startup device produces one nominal crossing and reaches the intended loaded current in the tested runs.


## A useful load and a fair sizing comparison

The bias node drives a SKY130 NMOS current output with W/L=0.42/20 Âµm and a declared 100 fF output capacitance. A resistor connects that output to the same VDD supply. There is no external bias source. At the intended operating point, the current is about 0.773 ÂµA over the tested 10 kÎ© to 1 MÎ© load range.

The saved comparison uses the same current target, a 5% band, at least 0.3 V output compliance and VDD at least 1.7 V. Readiness must occur within 1 ms after the ramp finishes and persist through the run for at least 2 ms. Both startup sizes pass all six load/ramp combinations. The six unassisted controls fail.

The wider startup device is faster with a 1 Âµs ramp. With a 1 ms ramp it uses about 6.7% more net source work to the same readiness point. This is a sizing trade-off in one declared model example. It is not a universal ranking. Source work includes the entire circuit and output load. We do not compare energy against the failed control.

Readiness is interpolated between samples. The measurement was checked against an analytic settling example. In four selected refinement runs, energy changed by less than 0.35% and readiness time by less than 0.3%. The following cell reads saved results; it does not rerun the complete sizing study.

![Loaded sizing comparison](results/loaded/loaded_tradeoff.png)


In [5]:
from loaded_startup import LOAD_STUDY
loaded = json.loads((LOAD_STUDY / 'measured_results.json').read_text())
print('Working loaded cases:', sum(r['meets_contract'] for r in loaded['cases']), '/', len(loaded['cases']))
for r in loaded['refined']:
    print('Width:', r['width_um'], 'um; ramp:', r['ramp_s'], 's; ready:', r['ready_time_s'], 's; source work:', r['source_energy_to_ready_J'], 'J')


Working loaded cases: 12 / 18
Width: 0.42 um; ramp: 1e-06 s; ready: 1.8141432613375988e-05 s; source work: 1.175832733047689e-10 J
Width: 0.42 um; ramp: 0.001 s; ready: 0.0009454444444444444 s; source work: 7.858326082458017e-09 J
Width: 1.0 um; ramp: 1e-06 s; ready: 1.635748919713083e-05 s; source work: 1.177132588157868e-10 J
Width: 1.0 um; ramp: 0.001 s; ready: 0.0009454444452222223 s; source work: 8.384270213294631e-09 J


## Reuse on another circuit and a healthy control

The second self-biased family is a conventional beta multiplier. Its NMOS width ratio is 4:1 at equal lengths, with an ideal 100 kÎ© source resistor. The PMOS mirror uses W/L=4/2 Âµm. The NMOS reference and output use 2/2 Âµm; the source-degenerated device uses 8/2 Âµm. Nodes a, b and s each have a declared 100 fF load. Only the source-degenerated NMOS gate is cut. The diode-connected reference remains intact.

The healthy control is an acyclic mirror. A 1 MÎ© resistor supplies a diode-connected NMOS reference; the output gate follows its bias node. Only the output gate is cut for the diagnostic. The reference voltage does not depend on the scan input, so its return curve is flat. This is a positive control, not a feedback-loop stability example. Its reference node has a 100 fF load.

Both examples use an NMOS output with a resistor to VDD, a zero-volt current sensor and a 100 fF output load. Junction geometry follows the main example, with default zero NRD/NRS. Both run through the same audit, intact-state check and readiness measurement. Their netlists come from `circuits.py`.

![Additional circuit connections](results/reuse/additional_circuits.png)

The next cell freshly runs both at TT, 27 Â°C, GMIN=1 fS, a 100 kÎ© load and a 1 Âµs ramp. Neither has an added startup branch. Each target comes from that circuit's intended loaded operating point. We do not rank energy across circuit families.


In [6]:
additional_live = []
for name, circuit in CIRCUITS.items():
    report = verify_circuit(circuit)
    assert report['status'] == 'completed'
    assert len(report['candidate_operating_points']) == 1
    assert report['startup_verdict'] == 'passes_declared_bench'
    additional_live.append(report)
    print(name, ':', report['startup_verdict'], '; target:', report['loaded_test']['nominal_target_A'], 'A')


beta_multiplier : passes_declared_bench ; target: 7.00457737e-07 A


resistor_biased_mirror : passes_declared_bench ; target: 1.24571592e-06 A


### Saved reuse study

The saved study covers two GMIN values (1 pS and 1 fS), two output loads (100 kÎ© and 1 MÎ©) and two supply ramps (1 Âµs and 1 ms). Both additional circuits show one confirmed candidate and pass all eight combinations each. The beta multiplier's unassisted startup is a measured outcome. Self-bias alone does not establish a startup failure in a particular model configuration.

The original inverse-Widlar regressions still show one or three candidates depending on GMIN. The unassisted cases fail the declared deadline; the assisted case passes. This section reads saved data. Regenerate it with `python qualify_reuse.py`, then `python plot_reuse.py`. These additional circuits have not received the main example's PVT or recovery qualification.

![Cross-circuit comparison](results/reuse/reuse_evidence.png)


In [7]:
reuse = json.loads((PROJECT / 'results/reuse/family_checks.json').read_text())
assert reuse['status'] == 'passed'
for name in CIRCUITS:
    cases = [r for r in reuse['cases'] if r['circuit'] == name]
    print(name, ':', sum(r['loaded_test']['meets_contract'] for r in cases), '/', len(cases), 'pass')
    print('Maximum intact-state mismatch (V):', max(p['max_state_difference_V'] for r in cases for p in r['candidate_operating_points']))


beta_multiplier : 8 / 8 pass
Maximum intact-state mismatch (V): 1.0551892526411422e-07
resistor_biased_mirror : 8 / 8 pass
Maximum intact-state mismatch (V): 2.7449591567529907e-06


## Qualification beyond the nominal example

A saved loaded-circuit study covers five corners at -20, 27 and 85 Â°C. Each condition's target is its intended high-state current, computed before the cold start. Both startup sizes pass all 15 tested conditions. The unassisted controls fail the 1 ms readiness deadline. This is convergence to the intended state, not a constant-current accuracy claim across PVT.

Both sizes also recover after the 12 tested supply-dip cases. The circuit first starts normally; no internal node is forced to create a working prehistory. Recovery energy is not compared because the accounting would also need the initial stored energy. Four additional unchanged-design cases at 10 and 60 Â°C use a 330 kÎ© load, 100 Âµs ramp and GMIN of 0.1 fS. They pass the declared contract.

The contact-resistance investigation remains bounded. Smaller nonzero NRD/NRS produced rejected DC data across the tested solvers and settings. Much larger NRD/NRS values produced valid scans. This supports a numerical-conditioning concern, but it is not a complete diagnosis. The supported entry point keeps the explicit zero-NRD/NRS scope and rejects unaudited DC results.


## Limits and rejected evidence

The supported scope uses default zero NRD/NRS, while intrinsic BSIM4 resistance remains present. Small nonzero-NRD/NRS trials invoked transient operating-point fallback and returned nonzero cut-gate current. We exclude those curves. Larger tested values produced valid scans, suggesting a numerical-conditioning concern without establishing its complete cause. The next cell checks a real rejected-output fixture.

The crossing finder includes exact sampled zeros and scan endpoints. Finite grids can still miss close crossings and unsampled tangent roots. Local return slopes are diagnostics, not a proof of dynamic stability for arbitrary circuits. The detailed nominal perturbations belong to the inverse-Widlar example. New adapters require a justified gate cut and an independently chosen intended state.

PVT and reuse targets are condition-specific intended operating points. Passing means convergence under the contract, not absolute current accuracy. The main high state draws about 113 ÂµA including the output branch. It is a verification example, not an optimized batteryless reference. No source-limited benefit, mismatch yield, extracted layout or silicon reliability is established.

The live notebook runs fresh nominal analyses. Wider studies are saved data with regeneration scripts. Fresh nominal execution has been checked on Windows and an Alpine Linux VM on the same computer. The wider saved PVT and recovery studies remain Windows results. Colab and macOS remain unverified.


In [8]:
negative = audit_return_case(PROJECT / 'results/fixtures/rejected_contact')
assert not negative['accepted']
print('Rejected fixture:', negative['reasons'])


Rejected fixture: ['Nonzero cut-gate current violates this unloaded gate-cut bench', 'Transient operating-point fallback was used during the DC run']


## Saved cross-platform check

The same numerical source files were used on Windows and in the Alpine Linux VM. All five fresh nominal cases agree in candidate count and startup verdict, including the two failing inverse-Widlar controls. The confirmed bias points and intended currents match at the recorded SPICE output precision. The Linux VM runs on the same physical computer, so this is cross-OS and cross-build evidence, not an independent-machine or second-simulator test.

Two additional matched slow-ramp runs check the sizing result. Both startup sizes pass the same current/compliance/deadline/hold contract on Linux. The larger size uses 6.693% more source energy there. The largest energy difference from the fresh Windows reference is 2.847e-06%. The comparison limits were 10 µV for bias points, 0.1% for intended current and 1% for the refined sizing time/energy.

These are saved portability results. `reproduce_short.py` and `portability_check.py` regenerate the simulations on each platform. `compare_platforms.py` compares saved records and does not rerun SPICE. The wider PVT and recovery studies remain Windows results. These differences describe agreement between the recorded runs, not bounds on simulation or physical error.


## References and license

- [Wang et al., Performance Verification of Start-Up Circuits in Reference Generators, 2012](https://class.ece.iastate.edu/vlsi2/docs/Papers%20Done/2012-08-MWSCAS-YW.pdf), Figs. 1, 2 and 8. The main circuit and startup connection follow this work with different models and declared dimensions.
- [Wang et al., Practical Methods for Verifying Removal of Trojan Stable Operating Points, 2013](https://class.ece.iastate.edu/vlsi2/docs/Papers%20Done/2013-05-ISCAS-YW.pdf), for verification methods and their limits.
- [ngspice manual](https://ngspice.sourceforge.io/docs/ngspice-manual.pdf), for GMIN and operating-point options.
- [Pinned upstream SKY130 models](https://github.com/google/skywater-pdk-libs-sky130_fd_pr/tree/f62031a1be9aefe902d6d54cddd6f59b57627436).

Project code uses Apache-2.0. Model copyright and licenses remain with the SkyWater PDK Authors. Prior Code-a-Chip entries informed the presentation and validation standard; their source code was not copied into this project.
